# FlashDance: Flash Attention Benchmarks

How much faster is Flash Attention vs vanilla attention? When does it become essential?

We compare:
- **Vanilla**: standard scaled dot-product attention (materializes full NxN matrix)
- **SDPA**: PyTorch's `scaled_dot_product_attention` (picks best available kernel)

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

from attention import vanilla_attention, sdpa_attention, MultiHeadAttention, check_backends
from benchmark import benchmark_attention, benchmark_backward, measure_time

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Quick sanity check

Make sure both implementations produce the same output.

In [ ]:
torch.manual_seed(42)
B, H, T, D = 2, 4, 128, 64
q = torch.randn(B, H, T, D, device=device)
k = torch.randn(B, H, T, D, device=device)
v = torch.randn(B, H, T, D, device=device)

out_vanilla = vanilla_attention(q, k, v, causal=True)
out_sdpa = sdpa_attention(q, k, v, causal=True)

diff = (out_vanilla - out_sdpa).abs().max().item()
print(f"Max absolute difference: {diff:.6f}")
print(f"Match: {'YES' if diff < 1e-3 else 'NO (but expected with different precision)'}")

## 2. Why Flash Attention exists

Standard attention computes the full NxN attention matrix:
- Memory: O(N^2) for the attention weights
- At N=4096 with batch=4, heads=8: that's 4 * 8 * 4096 * 4096 * 4 bytes = **2 GB** just for attention scores

Flash Attention tiles the computation so it never materializes the full matrix in GPU HBM.

In [ ]:
# show the O(N^2) memory growth
seq_lens = [128, 256, 512, 1024, 2048, 4096, 8192, 16384]
B, H = 4, 8

attn_matrix_mb = []
for n in seq_lens:
    # float32: 4 bytes per element
    size_bytes = B * H * n * n * 4
    attn_matrix_mb.append(size_bytes / 1024**2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(seq_lens)), attn_matrix_mb, color="#e74c3c")
ax.set_xticks(range(len(seq_lens)))
ax.set_xticklabels([str(s) for s in seq_lens])
ax.set_xlabel("Sequence length")
ax.set_ylabel("Attention matrix size (MB)")
ax.set_title(f"Memory for NxN attention matrix (batch={B}, heads={H}, float32)")
for i, mb in enumerate(attn_matrix_mb):
    label = f"{mb:.0f}MB" if mb < 1024 else f"{mb/1024:.1f}GB"
    ax.text(i, mb + max(attn_matrix_mb)*0.02, label, ha="center", fontsize=9)

plt.tight_layout()
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/memory_growth.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nAt seq_len=16384: {attn_matrix_mb[-1]/1024:.1f} GB just for attention scores!")

## 3. Forward pass benchmark

In [ ]:
seq_lengths = [128, 256, 512, 1024, 2048, 4096]

# reduce if on CPU (it will be slow)
if device == "cpu":
    seq_lengths = [128, 256, 512, 1024]

results = benchmark_attention(
    seq_lengths=seq_lengths,
    batch_size=4,
    n_heads=8,
    head_dim=64,
    device=device,
    repeats=10,
)

df = pd.DataFrame(results)
print(df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# speed comparison
ax = axes[0]
x = range(len(df))
width = 0.35
ax.bar([i - width/2 for i in x], df["vanilla_ms"], width, label="Vanilla", color="#e74c3c")
ax.bar([i + width/2 for i in x], df["sdpa_ms"], width, label="SDPA", color="#3498db")
ax.set_xticks(x)
ax.set_xticklabels(df["seq_len"])
ax.set_xlabel("Sequence length")
ax.set_ylabel("Time (ms)")
ax.set_title("Forward pass time")
ax.legend()

# speedup
ax = axes[1]
colors = ["#2ecc71" if s > 1 else "#e74c3c" for s in df["speedup"]]
ax.bar(x, df["speedup"], color=colors)
ax.axhline(y=1, color="gray", linestyle="--", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(df["seq_len"])
ax.set_xlabel("Sequence length")
ax.set_ylabel("Speedup (x)")
ax.set_title("SDPA speedup over vanilla")
for i, s in enumerate(df["speedup"]):
    ax.text(i, s + 0.05, f"{s:.2f}x", ha="center", fontsize=9)

# memory comparison
ax = axes[2]
ax.bar([i - width/2 for i in x], df["vanilla_peak_mb"], width, label="Vanilla", color="#e74c3c")
ax.bar([i + width/2 for i in x], df["sdpa_peak_mb"], width, label="SDPA", color="#3498db")
ax.set_xticks(x)
ax.set_xticklabels(df["seq_len"])
ax.set_xlabel("Sequence length")
ax.set_ylabel("Peak memory (MB)")
ax.set_title("Peak memory usage")
ax.legend()

plt.suptitle("Flash Attention vs Vanilla (forward pass)", fontsize=13)
plt.tight_layout()
plt.savefig("plots/forward_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Backward pass benchmark

Flash Attention also speeds up the backward pass. The recomputation trick avoids storing the full attention matrix for backprop.

In [ ]:
backward_lengths = [128, 256, 512, 1024, 2048]
if device == "cpu":
    backward_lengths = [128, 256, 512]

backward_results = benchmark_backward(
    seq_lengths=backward_lengths,
    batch_size=4,
    n_heads=8,
    head_dim=64,
    device=device,
    repeats=5,
)

bdf = pd.DataFrame(backward_results)
print(bdf.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = range(len(bdf))
width = 0.35

ax = axes[0]
ax.bar([i - width/2 for i in x], bdf["vanilla_backward_ms"], width, label="Vanilla", color="#e74c3c")
ax.bar([i + width/2 for i in x], bdf["sdpa_backward_ms"], width, label="SDPA", color="#3498db")
ax.set_xticks(x)
ax.set_xticklabels(bdf["seq_len"])
ax.set_xlabel("Sequence length")
ax.set_ylabel("Time (ms)")
ax.set_title("Backward pass time")
ax.legend()

ax = axes[1]
colors = ["#2ecc71" if s > 1 else "#e74c3c" for s in bdf["backward_speedup"]]
ax.bar(x, bdf["backward_speedup"], color=colors)
ax.axhline(y=1, color="gray", linestyle="--", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(bdf["seq_len"])
ax.set_xlabel("Sequence length")
ax.set_ylabel("Speedup (x)")
ax.set_title("SDPA backward speedup")
for i, s in enumerate(bdf["backward_speedup"]):
    ax.text(i, s + 0.05, f"{s:.2f}x", ha="center", fontsize=9)

plt.suptitle("Backward pass: Flash Attention vs Vanilla", fontsize=13)
plt.tight_layout()
plt.savefig("plots/backward_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Scaling: how time grows with sequence length

In [ ]:
# theoretical O(N^2) vs actual scaling
fig, ax = plt.subplots(figsize=(10, 6))

seq_lens = df["seq_len"].values
vanilla_times = df["vanilla_ms"].values
sdpa_times = df["sdpa_ms"].values

ax.plot(seq_lens, vanilla_times, "o-", color="#e74c3c", label="Vanilla (actual)", linewidth=2)
ax.plot(seq_lens, sdpa_times, "o-", color="#3498db", label="SDPA (actual)", linewidth=2)

# theoretical O(N^2) curve, scaled to match vanilla at first point
n2_curve = [(s / seq_lens[0])**2 * vanilla_times[0] for s in seq_lens]
ax.plot(seq_lens, n2_curve, "--", color="gray", alpha=0.5, label="O(N^2) reference")

ax.set_xlabel("Sequence length")
ax.set_ylabel("Time (ms)")
ax.set_title("Attention time scaling with sequence length")
ax.legend()
ax.set_yscale("log")
ax.set_xscale("log")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("plots/scaling.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. MultiHeadAttention module comparison

Test with the full MHA module (includes QKV projection and output projection).

In [ ]:
dim = 512
n_heads = 8

mha_vanilla = MultiHeadAttention(dim, n_heads, backend="vanilla").to(device)
mha_sdpa = MultiHeadAttention(dim, n_heads, backend="sdpa").to(device)

# copy weights so they compute the same thing
mha_sdpa.load_state_dict(mha_vanilla.state_dict())

mha_results = []
test_lens = [256, 512, 1024, 2048]
if device == "cpu":
    test_lens = [256, 512]

for seq_len in test_lens:
    x = torch.randn(4, seq_len, dim, device=device)

    t_vanilla = measure_time(mha_vanilla, x, warmup=3, repeats=10)
    t_sdpa = measure_time(mha_sdpa, x, warmup=3, repeats=10)

    speedup = t_vanilla["mean"] / t_sdpa["mean"]
    print(f"seq_len={seq_len}: vanilla={t_vanilla['mean']*1000:.2f}ms, sdpa={t_sdpa['mean']*1000:.2f}ms, speedup={speedup:.2f}x")

    mha_results.append({
        "seq_len": seq_len,
        "vanilla_ms": t_vanilla["mean"] * 1000,
        "sdpa_ms": t_sdpa["mean"] * 1000,
        "speedup": speedup,
    })

## 7. Available backends

Check which SDPA backends are available on this hardware.

In [ ]:
if torch.cuda.is_available():
    backends = check_backends()
    for name, available in backends.items():
        status = "available" if available else "not available"
        print(f"  {name}: {status}")
else:
    print("CUDA not available. SDPA falls back to:")
    if device == "mps":
        print("  - MPS-optimized math kernel")
    else:
        print("  - CPU math kernel")
    print("\nNote: Flash Attention kernel requires CUDA. On CPU/MPS, SDPA still")
    print("uses an optimized implementation but not the tiled IO-aware algorithm.")

## 8. Attention pattern visualization

What does the attention matrix actually look like? The causal mask creates a triangular pattern.

In [ ]:
from attention import vanilla_attention_with_scores
from visualize import plot_causal_vs_bidirectional, plot_all_heads

# causal vs bidirectional
plot_causal_vs_bidirectional(seq_len=32)

# all heads
torch.manual_seed(0)
q = torch.randn(1, 8, 64, 64, device=device)
k = torch.randn(1, 8, 64, 64, device=device)
v = torch.randn(1, 8, 64, 64, device=device)
_, attn = vanilla_attention_with_scores(q, k, v, causal=True)
plot_all_heads(attn, save_path="plots/attention_heads.png")

## 9. Precision comparison

Does using float16 or bfloat16 change the output much? Flash Attention is designed to work well with reduced precision.

In [ ]:
from attention import compare_dtypes

dtype_results = compare_dtypes(seq_len=512, device=device)
for dtype_name, diffs in dtype_results.items():
    print(f"\n{dtype_name}:")
    for k, v in diffs.items():
        print(f"  {k}: {v:.6f}")

## Key takeaways

**Flash Attention is about memory, not just speed.**

Standard attention materializes an NxN matrix in GPU memory. At seq_len=4096, that's gigabytes. Flash Attention tiles the computation so it never stores the full matrix. This means you can train on longer sequences with the same GPU.

**The speedup grows with sequence length.**

At short sequences (128-256 tokens), the overhead of the tiling might not help much. At 2K+ tokens, Flash Attention is significantly faster because memory bandwidth becomes the bottleneck.

**Use `torch.nn.functional.scaled_dot_product_attention`.**

Since PyTorch 2.0, you get Flash Attention for free through SDPA. It automatically picks the best available kernel. No need to install the `flash-attn` package separately (though that package may offer newer features).

**The backward pass matters too.**

Flash Attention recomputes attention during backprop instead of storing it. This trades a bit of compute for a lot of memory savings. For training, this is often the bigger win.